In [1]:
#  Paso 1: Importar librerías
# =============================================================================
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import numpy as np

In [5]:
#  Paso 2: Cargar datos y definir los 5 DESCRIPTORES MOLECULARES seleccionados
# =============================================================================

# --- Configuración Inicial ---
# Ruta a tu archivo CSV de entrenamiento de Descriptores Moleculares
ruta_archivo_csv = r"C:\Users\benja\Desktop\BD PAMPA\Calculos descriptores moleculares\training.csv" 

# Nombre de la columna de la clase/actividad.
columna_clase = "Actividad"

# Lista de los 5 descriptores seleccionados por consenso.
atributos_seleccionados = [
    'LOGPcons',
    'piPC05',
    'CATS2D_07_AP',
    'B06[C-C]',
    'Eig12_EA(dm)'
]
print(f"Cargando datos desde: {ruta_archivo_csv}")

# --- Carga y Preparación de Datos ---
df_completo = pd.read_csv(ruta_archivo_csv)
# No es necesario limpiar columnas aquí porque el archivo se generó con Python
X = df_completo[atributos_seleccionados]
y = df_completo[columna_clase]

print(f"Se usarán {X.shape[1]} atributos seleccionados para {X.shape[0]} moléculas.")


Cargando datos desde: C:\Users\benja\Desktop\BD PAMPA\Calculos descriptores moleculares\training.csv
Se usarán 5 atributos seleccionados para 4357 moléculas.


In [6]:
#  Paso 3: Definir los modelos a evaluar (los mismos que antes)
# =============================================================================
modelos = {
    "Árbol de Decisión (J48)": DecisionTreeClassifier(random_state=42),
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=42),
    "k-NN (IBk, k=5)": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}


In [7]:
#  Paso 4: Entrenar y Evaluar con Validación Cruzada (10-folds)
# =============================================================================
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
resultados_dm = {}

print("\n--- Iniciando Evaluación de Modelos (Descriptores Moleculares) ---")
for nombre, modelo in modelos.items():
    scores_accuracy = cross_val_score(modelo, X, y, cv=cv, scoring='accuracy')
    scores_roc_auc = cross_val_score(modelo, X, y, cv=cv, scoring='roc_auc')
    
    resultados_dm[nombre] = {
        'Accuracy': np.mean(scores_accuracy),
        'ROC AUC': np.mean(scores_roc_auc)
    }
    print(f"  > Evaluado: {nombre}")
print("--- Evaluación Completada ---\n")


--- Iniciando Evaluación de Modelos (Descriptores Moleculares) ---
  > Evaluado: Árbol de Decisión (J48)
  > Evaluado: Regresión Logística
  > Evaluado: k-NN (IBk, k=5)
  > Evaluado: Random Forest
  > Evaluado: SVM
--- Evaluación Completada ---



In [9]:
#  Paso 5: Mostrar y Guardar resultados
# =============================================================================
df_resultados_dm = pd.DataFrame.from_dict(resultados_dm, orient='index')
print("📊 Tabla Comparativa de Rendimiento (Descriptores Moleculares):")
print(df_resultados_dm.round(3))



📊 Tabla Comparativa de Rendimiento (Descriptores Moleculares):
                         Accuracy  ROC AUC
Árbol de Decisión (J48)     0.653    0.657
Regresión Logística         0.733    0.765
k-NN (IBk, k=5)             0.713    0.757
Random Forest               0.715    0.761
SVM                         0.738    0.793


In [10]:
#PASO 6: GUARDAR LOS RESULTADOS EN UN ARCHIVO (NUEVO)
# =============================================================================
nombre_archivo_salida = 'resultados_modelos_dm.csv'
df_resultados_dm.to_csv(nombre_archivo_salida)

print(f"\n✅ ¡Tabla de resultados guardada exitosamente en el archivo '{nombre_archivo_salida}'!")


✅ ¡Tabla de resultados guardada exitosamente en el archivo 'resultados_modelos_dm.csv'!
